# 04 System integration

Owner: **Larissa**

This notebook ties the three models together and demonstrates the transaction flow from the assignment diagram:

1. A **face** image must be recognised as a registered user.
2. That user **requests a prediction**.
3. A **voice** sample must verify the **same** user. Only then is the product model called.

It uses the real artifacts: `models/face_model.joblib` (Maurice), `voice_model.joblib` (Kudakwashe) and `product_model.joblib` (Patrick). If one is missing, the cells stop with a message that says who needs to supply it. All outputs come from running the real code; nothing is typed in by hand.

In [ ]:
from pathlib import Path
import sys
import warnings
from argparse import Namespace

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "main.py").exists(), "Run from the project root or notebooks directory"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import sklearn
import pandas as pd
import main as app
import run_demo

DATA, MODELS = ROOT / "data", ROOT / "models"
args = Namespace(models=MODELS, registry=DATA / "processed/user_registry.csv",
                 customers=DATA / "processed/customer_inference.csv")
print("scikit-learn", sklearn.__version__)

## 1. Registry

`user_registry.csv` maps each consenting team member (`user_id`, identical to the class label used by the face and voice models) to the customer record they act as in this simulation. The assignment is an explicit, arbitrary choice for the demo, not real customer data. The four customers are ones the product model never saw in training, so its predictions are not inflated by memorisation.

In [ ]:
registry = pd.read_csv(args.registry, dtype=str)
display(registry)
assert not registry.user_id.duplicated().any(), "user_id must be unique"
assert not registry.customer_id.duplicated().any(), "each user needs a different customer"

## 2. Validate artifacts and shared interfaces

Checks every file the CLI needs, that the models' class labels are all registered users, and that every registered customer has exactly one pre-purchase row for the product model.

In [ ]:
needed = ["face_model.joblib", "voice_model.joblib", "product_model.joblib", "model_metadata.json"]
missing = [n for n in needed if not (MODELS / n).exists()]
assert not missing, (f"Missing in models/: {missing}. face_model.joblib is created by notebook 02 "
                     "(Maurice) and is listed in .gitignore; commit it with `git add -f`.")

ctx = app.load_context(args)           # also checks status == 'trained', thresholds, feature columns
meta, reg = ctx["metadata"], ctx["registry"]

for part in ("face", "voice"):
    model = ctx[f"{part}_model"]
    labels = set(map(str, model.classes_))
    print(f"{part}: {type(model).__name__} | classes={sorted(labels)} | threshold={meta[part]['threshold']:.3f}")
    assert not (labels - set(reg)), f"{part} model has labels that are not in the registry: {labels - set(reg)}"
print("face probability gate:", ctx["face_gate"], "(the face model also applies its own cosine-similarity gate)")

customers, cols = ctx["customers"], meta["product"]["feature_columns"]
rows = []
for user, cid in reg.items():
    r = customers[customers.customer_id == cid]
    assert len(r) == 1, f"{user}: expected exactly 1 inference row for customer {cid}, found {len(r)}"
    assert not r[cols].isna().any().any(), f"{user}: missing values in product features"
    proba = ctx["product_model"].predict_proba(r[cols]).max()
    rows.append({"user": user, "customer_id": cid, "recommendation": ctx["product_model"].predict(r[cols])[0],
                 "model_confidence": round(float(proba), 3)})
display(pd.DataFrame(rows))
print("Artifacts, labels and customer rows are consistent.")

## 3. Scripted scenarios

Each attempt goes through exactly the function the CLI uses (`main.attempt`):

| Scenario | Face | Voice | Expected |
|---|---|---|---|
| authorized_* | registered member (test photo) | same member | granted, product shown |
| unknown_face | stranger from `data/raw/images/unauthorized/` | member | denied at the **face** check, voice never processed |
| blank_image | plain white image | member | denied at the **face** check (no face) |
| unknown_voice | member | stranger recording | denied at the **voice** check |
| silent_audio | member | one second of silence | denied at the **voice** check |
| mismatched_voice | member A | member B (also registered) | denied at the **identity-match** check |

A scenario whose sample file is missing is marked `SKIPPED`, never `PASS`. Record a short clip of a non-member saying the phrase and save it as `data/raw/audio/unauthorized/unauthorized_voice.wav` to enable `unknown_voice`. The voice recordings in the authorized scenarios were also used to train or calibrate the voice model (there are only two per speaker), so they show that the flow works, not how accurate it is on new recordings.

In [ ]:
scenarios = pd.read_csv(DATA / "processed/demo_scenarios.csv", dtype=str)
results = run_demo.run_scenarios(ctx, scenarios, root=ROOT)
results.to_csv(ROOT / "reports/integration_results.csv", index=False)
display(results)

## 4. Safety checks and integration summary

In [ ]:
ran = results[results.status != "SKIPPED"]
print("Scenario status counts:", results.status.value_counts().to_dict())
assert (results.status != "FAIL").all(), "At least one scenario behaved differently from expected"

denied = ran[ran.actual.str.startswith("denied")]
assert not denied.detail.isin(list(ctx["product_model"].classes_)).any(), "A denied attempt returned a product"

authorized = ran[ran.expected.str.startswith("granted")]
unauthorized = ran[ran.expected.str.startswith("denied")]
summary = pd.Series({
    "authorized attempts": len(authorized),
    "authorized granted (correct)": int(authorized.actual.str.startswith("granted").sum()),
    "unauthorized attempts": len(unauthorized),
    "unauthorized granted (false accepts)": int(unauthorized.actual.str.startswith("granted").sum()),
}, name="count")
display(summary.to_frame())
print("These counts come from a handful of attempts, so they demonstrate the logic. They do not "
      "estimate real-world false accept/reject rates.")

## 5. How reliable is the voice check?

The voice model is the weakest link, so it gets its own check. This cell needs no audio libraries: `audio_features.csv` already stores each recording's speaker embedding, and the cell passes those through the real voice model with the real threshold. It shows (a) whether each recording is identified correctly and (b) what would happen if a member's face were presented with another member's validation recording (a mismatch attack).

In [ ]:
import numpy as np

voice_model, v_threshold = ctx["voice_model"], meta["voice"]["threshold"]
v_cols = meta["voice"]["feature_columns"]
features = pd.read_csv(DATA / "processed/audio_features.csv")
orig = features[features.augmentation == "original"].reset_index(drop=True)

rows = []
for i, r in orig.iterrows():
    p = voice_model.predict_proba(orig.loc[[i], v_cols].to_numpy(dtype=np.float32))[0]
    top = int(p.argmax())
    rows.append({"speaker": r.user_id, "split": r.split, "recording": r.source_id.split("/")[1],
                 "predicted": voice_model.classes_[top], "probability": round(float(p[top]), 3),
                 "correct": voice_model.classes_[top] == r.user_id,
                 "above_threshold": bool(p[top] >= v_threshold)})
voice_table = pd.DataFrame(rows)
display(voice_table)

val = orig[orig.split == "validation"].reset_index(drop=True)
attacks, accepted = 0, []
for i, r in val.iterrows():
    p = voice_model.predict_proba(val.loc[[i], v_cols].to_numpy(dtype=np.float32))[0]
    top = int(p.argmax())
    for face_user in voice_model.classes_:
        if face_user == r.user_id:
            continue
        attacks += 1
        if p[top] >= v_threshold and voice_model.classes_[top] == face_user:
            accepted.append((face_user, r.user_id, r.source_id.split("/")[1], round(float(p[top]), 3)))

print(f"Voice threshold {v_threshold:.3f}; chance level for {len(voice_model.classes_)} speakers is "
      f"{1 / len(voice_model.classes_):.3f}.")
print(f"Mismatch attacks on validation recordings (face of A + recording of B): "
      f"{len(accepted)} accepted out of {attacks}.")
for face_user, voice_user, rec, prob in accepted:
    print(f"  accepted: face={face_user}, voice={voice_user} ({rec}), voice model said {face_user} at {prob}")
print(f"Genuine recordings identified correctly: {int(voice_table.correct.sum())} of {len(voice_table)} "
      f"(validation only: {int(voice_table[voice_table.split == 'validation'].correct.sum())} of {len(val)}).")

**Reading this honestly.** The voice model's probabilities are low (for four speakers, 0.25 is chance) and the threshold sits only slightly above chance, so it separates enrolled voices from each other only weakly and it cannot reject unknown speakers reliably. With only two recordings per speaker there is no independent test set, and no unknown-speaker recordings exist to measure false acceptance of strangers. Treat the voice step as a demonstration of the logic and collect more recordings before trusting it.

## 6. Live walk-through of one full transaction and one denied attempt

In [ ]:
authorized_row = scenarios[scenarios.scenario == "authorized_larissa"].iloc[0]
denied_row = scenarios[scenarios.scenario == "mismatched_voice"].iloc[0]

for title, row in (("AUTHORIZED", authorized_row), ("DENIED (other member's voice)", denied_row)):
    print(f"\n--- {title} ---")
    r = app.attempt(ctx, ROOT / row.face, ROOT / row.voice)
    print("Result:", {k: r[k] for k in ("outcome", "stage", "identity", "prediction", "reason")})

## 7. Run the tests

In [ ]:
import subprocess
out = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"],
                     cwd=ROOT, capture_output=True, text=True)
print(out.stdout[-1500:], out.stderr[-2500:])
assert out.returncode == 0

## 8. How the multimodal logic works

- **Two independent factors.** The face model and the voice model are trained separately on different data (face embeddings from SFace, speaker embeddings from ECAPA-TDNN) and each outputs class scores for the four registered users.
- **Order matters.** The face is checked first. If it fails (no face, several faces, unknown person), the voice file is never processed and the product model is never called. After the face passes, the user requests a prediction and a voice sample must confirm it.
- **Face rejection.** The face model rejects a face unless its classifier probability is at least 0.5 **and** its cosine similarity to the predicted member's average embedding reaches `face.threshold`. A rejected face returns all-zero scores, which the CLI treats as denial.
- **Same person required.** Both checks must name the *same* registered user, so a valid voice from a different enrolled member is rejected at the identity-match step.
- **Fail closed.** An unreadable or silent file counts as a denial, never a crash and never a prediction.
- **The product model only runs for the verified customer.** It receives that customer's single pre-purchase feature row and returns the recommended product. Its held-out accuracy is 31.8% against 20% chance, so treat the recommendation as weak.
- **Every attempt is logged** to `reports/access_log.csv` with the stage at which it stopped.

### Limitations (state these in the report)
- Very few samples per person: confidence scores and thresholds are uncertain.
- The voice threshold is close to chance level; no unknown-speaker recordings exist to measure false acceptance.
- No liveness detection: a printed photo or a replayed recording could be presented.
- The registry-to-customer links are a simulation choice, not real accounts.

## 9. Before you submit
- [ ] `models/face_model.joblib` and `data/processed/image_features.csv` are committed (`git add -f`)
- [ ] everyone uses the same scikit-learn version and `models/product_model.joblib` loads on every machine
- [ ] `data/raw/audio/unauthorized/unauthorized_voice.wav` recorded so `unknown_voice` runs
- [ ] `reports/integration_results.csv` and `reports/access_log.csv` generated from a real run
- [ ] `contributions.md` and `final_report.md` completed and exported to `final_report.pdf`; repo and video links added